In [ ]:
# Purpose - makes fields/spectra movie (newer)

findex = 6 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
boilerplate = run.boilerplate #run.boilerplate + f"tstep = {run.steps[0]}"

my_steps = run.steps[:-1]

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields")

In [ ]:
def compute_Az(roi):    
    return np.cumsum(roi.bx, axis=0) - np.cumsum(roi.by, axis=1) #properly should multiply this by the spacing

In [ ]:
print(run.cc * run.output_interval / run.c_omp * 0.1)

In [ ]:

# Plot Domain and Spectrum

counter = 0
gauge_integral = 0

for abs_ts in range(10, 17): #[14, 15, 16, 17]:
    plt.close()

    rect = [0, run.xlim, 0, run.ylim]
    #rect = [15, 65, 120, 130]

    roi = fq_cut(d.fields, abs_ts - my_steps[0], rect)

    cs = roi

    # compute gauge 
    corner_ez = roi.ez[0, 0]
    

    Az = compute_Az(roi)
    Azp = Az + gauge_integral

    plot_fq3(cs, cs.dens, vabsmax = 3)
    #plot_fq3(cs, Azp) #Az' = Az + gauge_integral (note sign flip relative to what Robbie sent)

    levels = [8] #[1]
    plt.contour(Azp.x, Azp.y, Azp, levels=levels, colors = 'white')

    plt.suptitle(run.boilerplate + f"\n $t \Omega$ =  {run.gyrofreq * abs_ts:.2f} ({abs_ts})")

    plt.xlim(rect[:2])
    plt.ylim(rect[2:])

    plt.show()

    gauge_integral -= run.cc * run.output_interval * corner_ez
    print(f"gauge integral is {gauge_integral.values}")
    save_movie_frame(counter)
    counter += 1
